# Verify a design against Cameo requirements

Verification driven by **requirements extracted from a Cameo (`.mdzip`) model in Istari**. Nothing in the pass/fail logic is hard-coded: the limits come out of the SysML requirement text, so the authoritative source of truth stays in the systems model.

1. Read `requirements.json` from a Cameo extraction already stored in Istari
2. Check a design parameter set against those requirement limits
3. Upload the outputs and log a `FAILED` entry
4. Fix the violation, re-run, and log `SUCCESS`
5. Attach the passing report to the design model as an artifact

Companion to [Scenario A quickstart](workflow_log_scenario_a_simple.ipynb), which uses the same workflow-log mechanics with a synthetic stress check instead of real requirements.

### Prerequisites

Run these in a terminal from the cookbook root **before** starting this notebook — the kernel has to exist before the notebook can attach to it:

```bash
uv sync --group dev
```

```bash
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Then pick the **Python (istari-client-cookbook)** kernel. Only the `dev` group is needed.

Also required:

- **Registry Service > 10.17.3** (2026-05 release or later)
- Credentials in [`samples/.env`](../.env): `ISTARI_REGISTRY_URL`, `ISTARI_PERSONAL_ACCESS_TOKEN`
- **Experimental features** enabled in the web app (Istari Digital → **Application Settings**) so the **Workflow log** tab is visible
- A Cameo `.mdzip` model on your instance that has already been extracted — see below. On a new instance, set `REQUIREMENTS_RESOURCE_ID` in §2 rather than relying on the file name

### Where `requirements.json` comes from

The Cameo integration writes it as a job product. If your instance has no extraction yet, produce one with a Cameo agent:

```python
job = client.add_job(model_id=MDZIP_MODEL_ID, function="@istari:extract", tool_name="dassault_cameo")
```

Poll until `COMPLETED`, and `requirements.json` appears among the model's artifacts. [Cameo requirements extraction + tag update](../cameo_extract_and_update_notebook%20-%20demo%20(sdk%20only).ipynb) walks through that end to end, including writing values back into the model with `@istari:update_tags`. This notebook only *consumes* the extraction, so it needs no Cameo agent of its own.

## 1. Connect

`Client` reads models, artifacts, and files; `V3Client` handles workflow outputs and workflow log entries.

In [ ]:
import json
import os
import re
from pathlib import Path

from dotenv import load_dotenv

from istari_helpers import commit_changes

from istari_digital_client import Client, Configuration, V3Client
from istari_digital_client.v3.models import WorkflowLogEntryCreateDto

load_dotenv("../.env")
REGISTRY_URL = os.environ["ISTARI_REGISTRY_URL"]

config = Configuration(
    registry_url=REGISTRY_URL,
    registry_auth_token=os.environ["ISTARI_PERSONAL_ACCESS_TOKEN"],
)
client = Client(config)   # models, artifacts, files, systems
v3 = V3Client(config)     # workflow outputs + workflow log

UI_URL = REGISTRY_URL.rstrip("/").replace("//fileservice-v2.", "//")

print("Registry:", client.check_compatibility().server_version)
print("Web app:", UI_URL)

## 2. Read the requirements out of Istari

Set `REQUIREMENTS_RESOURCE_ID` to point straight at the extraction. Resource ids are portable; file names are not, so this is the reliable way to move the notebook between instances. It accepts either:

- the **`requirements.json` artifact** resource id, or
- the **`.mdzip` model** resource id — the newest `requirements.json` extraction on that model is used

Leave it as `None` to fall back to scanning for a Cameo model by file name. If that scan comes up empty it prints every `requirements.json` artifact it did find, with the id to paste back in — so a fresh instance takes one run to configure.

`parse_limits` pulls the numeric bounds out of the SysML requirement text. Three phrasings are recognised, covering both an exact nominal and a ceiling:

| Requirement text | Parsed as |
|---|---|
| *"shall be 9 inches"* | exact nominal, `= 9 in` |
| *"shall not exceed 110°C"* / *"up to 70°C"* | upper bound, `<= 110 °C` |
| *"shall be between 26 mm and 30 mm"* | range, `26-30 mm` |

Units are normalised to `in`, `°C`, and `mm`. Requirements with no number — narrative parents like *Environmental Conditions*, or empty placeholders — are skipped.

In [ ]:
from istari_digital_client.v3.models.resource_type_dto import ResourceTypeDto

# Preferred: point straight at the extraction (artifact resource id, or .mdzip model resource id).
REQUIREMENTS_RESOURCE_ID = None

# Fallback used only when the id above is None.
MDZIP_NAME = "HVMC.mdzip"  # set to your Cameo file, or use the id above
MAX_MODELS_SCANNED = 200
MIN_BOUNDED_REQUIREMENTS = 3  # skip stale extractions from before the requirements existed


_UNIT = r"(?:inches|inch|in|mm|\u00b0\s*C|degC|C)"
_RANGE_RE = re.compile(
    rf"between\s+(?P<low>[\d.]+)\s*(?:{_UNIT})?\s+and\s+(?P<high>[\d.]+)\s*(?P<unit>{_UNIT})\b",
    re.IGNORECASE)
_MAX_RE = re.compile(
    rf"(?:shall not exceed|up to|no more than|at most)\s+(?P<high>[\d.]+)\s*(?P<unit>{_UNIT})\b",
    re.IGNORECASE)
_EXACT_RE = re.compile(
    rf"shall be\s+(?P<value>[\d.]+)\s*(?P<unit>{_UNIT})\b",
    re.IGNORECASE)
TOLERANCE = 1e-9  # exact-nominal requirements compare with a float epsilon


def normalize_unit(raw):
    """Collapse the spellings Cameo authors use into one unit label."""
    unit = raw.replace(" ", "").lower()
    if unit in ("inches", "inch", "in"):
        return "in"
    if unit in ("\u00b0c", "degc", "c"):
        return "\u00b0C"
    return unit


def parse_limits(text):
    """Return (low, high, unit) from SysML requirement text, or None if not numeric.

    Either bound may be None, meaning unbounded on that side; low == high means the
    requirement states an exact nominal value.
    """
    match = _RANGE_RE.search(text)
    if match:
        return float(match["low"]), float(match["high"]), normalize_unit(match["unit"])
    match = _MAX_RE.search(text)
    if match:
        return None, float(match["high"]), normalize_unit(match["unit"])
    match = _EXACT_RE.search(text)
    if match:
        value = float(match["value"])
        return value, value, normalize_unit(match["unit"])
    return None


def describe_limits(low, high, unit):
    """Readable window for a parsed requirement."""
    if low is None:
        return f"<= {high:g} {unit}"
    if low == high:
        return f"= {low:g} {unit}"
    return f"{low:g}-{high:g} {unit}"


def bounded_count(raw):
    """How many requirements in this extraction carry machine-checkable limits."""
    return sum(1 for r in json.loads(raw.decode("utf-8")) if parse_limits(r.get("text") or ""))


def newest_requirements_artifact(model_id):
    """Newest requirements.json artifact on a Cameo model, or None."""
    matches = [
        a for a in client.list_model_artifacts(model_id, size=50).items
        if a.file and a.file.name == "requirements.json"
    ]
    return matches[-1] if matches else None


def load_requirements(resource_id):
    """Read the extraction behind a model or artifact resource id.

    Returns (requirements, raw_bytes, source) where source records the provenance.
    """
    resource = v3.get_resource(resource_id=resource_id)

    if resource.resource_type == ResourceTypeDto.MODEL:
        artifact = newest_requirements_artifact(resource_id)
        if artifact is None:
            raise RuntimeError(
                f"Model {resource.name!r} ({resource_id}) has no requirements.json artifact. "
                'Run @istari:extract with tool_name="dassault_cameo" on it first.'
            )
    elif resource.name == "requirements.json":
        artifact = client.get_artifact(resource_id)
    else:
        raise RuntimeError(
            f"Resource {resource_id} is {resource.name!r}. Pass the requirements.json artifact "
            "resource id, or the .mdzip model resource id."
        )

    revision = artifact.file.revisions[-1]
    raw = revision.read_bytes()
    cameo_model = client.get_model(artifact.model_id) if artifact.model_id else None
    extract_job = next(
        (s.resource_id for s in (revision.sources or [])
         if getattr(s, "resource_type", None) == "Job"),
        None,
    )
    source = {
        "cameo_model": cameo_model.file.name if cameo_model and cameo_model.file else None,
        "model_id": artifact.model_id,
        "artifact_id": artifact.id,
        "revision_id": revision.id,
        "extract_job_id": extract_job,
    }
    return json.loads(raw.decode("utf-8")), raw, source


def scan_for_extraction(mdzip_name):
    """Look up an extraction by Cameo file name.

    Returns (artifact_id, candidates); candidates lists every extraction seen so the
    failure message can show what this instance actually has.
    """
    candidates = []
    for scanned, model in enumerate(client.list_models(size=100).iter_items()):
        if scanned >= MAX_MODELS_SCANNED:
            break
        if not model.file or not (model.file.name or "").lower().endswith(".mdzip"):
            continue
        artifact = newest_requirements_artifact(model.id)
        if artifact is None:
            continue
        bounded = bounded_count(artifact.file.revisions[-1].read_bytes())
        candidates.append((model.file.name, artifact.id, bounded))
        if model.file.name == mdzip_name and bounded >= MIN_BOUNDED_REQUIREMENTS:
            return artifact.id, candidates
    return None, candidates


resource_id = REQUIREMENTS_RESOURCE_ID
if resource_id is None:
    resource_id, candidates = scan_for_extraction(MDZIP_NAME)
    if resource_id is None:
        listing = "\n".join(
            f"    {name}  ->  REQUIREMENTS_RESOURCE_ID = {artifact_id!r}  ({bounded} bounded)"
            for name, artifact_id, bounded in candidates
        ) or "    (no requirements.json extractions found on this instance)"
        raise RuntimeError(
            f"No extraction matched MDZIP_NAME={MDZIP_NAME!r} with at least "
            f"{MIN_BOUNDED_REQUIREMENTS} bounded requirements. Set REQUIREMENTS_RESOURCE_ID to "
            f"one of these:\n{listing}\n"
            'Or run @istari:extract with tool_name="dassault_cameo" on your Cameo model first.'
        )

requirements, REQUIREMENTS_BYTES, REQ_SOURCE = load_requirements(resource_id)

print(f"Cameo model:  {REQ_SOURCE['cameo_model']}  (model {REQ_SOURCE['model_id']})")
print(f"Requirements: artifact {REQ_SOURCE['artifact_id']}, revision {REQ_SOURCE['revision_id']}")
print(f"Extract job:  {REQ_SOURCE['extract_job_id']}")
print(f"{len(requirements)} requirements extracted; numerically bounded ones:\n")

for req in requirements:
    limits = parse_limits(req.get("text") or "")
    if limits:
        low, high, unit = limits
        name = (req.get("name") or "").strip()
        print(f"  [{req['req_id']:<9}] {name:<48} {describe_limits(low, high, unit)}")

## 3. Create the system for the design under test

The design is a plain JSON parameter set — the numbers a CAD build or an analysis would produce — tracked as one file on one configuration. Each run creates a fresh system; the teardown cell at the bottom archives it.

`commit_changes` (from [`istari_helpers.py`](istari_helpers.py)) snapshots the configuration and advances the baseline tag. A workflow log entry can only reference a configuration that is in the branch history, so this is required here and again after every design revision.

In [ ]:
from istari_digital_client.v2.models.new_system import NewSystem
from istari_digital_client.v2.models.new_system_configuration import NewSystemConfiguration
from istari_digital_client.v2.models.new_tracked_file import NewTrackedFile
from istari_digital_client.v2.models.tracked_file_specifier_type import TrackedFileSpecifierType

SYSTEM_NAME = "Cameo Requirements Verification"
CONFIG_NAME = "requirements-check"

# Coldplate dimensions are on nominal; the internal components run at 118 C, over the
# 110 C ceiling in requirement 3.6.1.1.2 - the one initial violation.
INITIAL_DESIGN = {
    "COLDPLATE_1": {"length_in": 9.0, "width_in": 3.4, "height_in": 0.94},
    "THERMAL": {"coolant_temp_c": 65.0, "component_temp_c": 118.0, "power_module_temp_c": 132.0},
}

work = Path("_cameo_req_run")
work.mkdir(exist_ok=True)
design = work / "hvmc-design.json"
design.write_text(json.dumps(INITIAL_DESIGN, indent=2))

model = client.add_model(
    path=design,
    display_name=design.name,
    description="HVMC coldplate + thermal budget - initial design",
)
system = client.create_system(
    NewSystem(
        name=SYSTEM_NAME,
        description=f"Design verified against requirements from {REQ_SOURCE['cameo_model']}",
    )
)
configuration = client.create_configuration(
    system_id=system.id,
    new_system_configuration=NewSystemConfiguration(
        name=CONFIG_NAME,
        tracked_files=[
            NewTrackedFile(
                specifier_type=TrackedFileSpecifierType.LATEST,
                file_id=model.file.id,
            )
        ],
    ),
)

SYSTEM_ID = system.id
CONFIG_ID = configuration.id
MODEL_ID = model.id
FILE_ID = model.file.id

commit_changes(client, SYSTEM_ID, CONFIG_ID)
print("Created system:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 4. Verify the design against the requirements

`PARAM_FOR_REQ` is the verification matrix: which design parameter satisfies which Cameo requirement. Six of the extracted requirements are covered — three coldplate dimensions and three temperature ceilings. Keys are matched with surrounding whitespace stripped, because `"Extended Requirement Coolant Temperature "` ships from Cameo with a trailing space. Everything else — the limits, the requirement IDs on each result — comes from the extraction, so editing a bound in Cameo and re-extracting changes the verdict here without touching this notebook.

Each run writes three output files, including the exact requirement set it was checked against.

In [ ]:
# Which design parameter each Cameo requirement constrains.
PARAM_FOR_REQ = {
    # Physical Characteristics :: Coldplate Dimensions
    "Extended Requirement Coldplate Length": ("COLDPLATE_1", "length_in"),
    "Extended Requirement Coldplate Width": ("COLDPLATE_1", "width_in"),
    "Extended Requirement Coldplate Height": ("COLDPLATE_1", "height_in"),
    # Environmental Conditions :: Natural Environment :: Temperture
    "Extended Requirement Coolant Temperature": ("THERMAL", "coolant_temp_c"),
    "Extended Requirement Component Temperature": ("THERMAL", "component_temp_c"),
    "Extended Requirement Power Module Temperature": ("THERMAL", "power_module_temp_c"),
}


def run_checks(design_bytes, out_dir):
    """Verify every mapped requirement, write the outputs, return (verdict, paths)."""
    values = json.loads(design_bytes)
    checks = []

    for req in requirements:
        target = PARAM_FOR_REQ.get((req.get("name") or "").strip())
        limits = parse_limits(req.get("text") or "")
        if target is None or limits is None:
            continue  # narrative or unmapped requirement - not machine-verifiable here

        part, parameter = target
        value = values[part][parameter]
        low, high, unit = limits
        passed = ((low is None or value >= low - TOLERANCE)
                  and (high is None or value <= high + TOLERANCE))
        window = describe_limits(low, high, unit)

        checks.append({
            "req_id": req["req_id"],
            "requirement": (req.get("name") or "").strip(),
            "parameter": f"{part}.{parameter}",
            "value": value,
            "unit": unit,
            "limits": [low, high],
            "passed": passed,
            "detail": f"{value:g} {unit} against {window}",
        })

    if not checks:
        raise RuntimeError(
            "No requirement matched PARAM_FOR_REQ. Extracted names: "
            f"{[r['name'] for r in requirements]}"
        )

    verdict = "SUCCESS" if all(check["passed"] for check in checks) else "FAILED"

    out_dir.mkdir(parents=True, exist_ok=True)
    results_path = out_dir / "results.json"
    report_path = out_dir / "report.txt"
    requirements_path = out_dir / "requirements.json"

    results_path.write_text(json.dumps({
        "verdict": verdict,
        "requirements_source": REQ_SOURCE,
        "checks": checks,
    }, indent=2))
    report_path.write_text(
        "".join(
            f"{'PASS' if check['passed'] else 'FAIL'}  [{check['req_id']:<9}] "
            f"{check['parameter']:<30}{check['detail']}\n"
            for check in checks
        )
    )
    requirements_path.write_bytes(REQUIREMENTS_BYTES)  # what we verified against

    print(report_path.read_text(), end="")
    print("Verdict:", verdict)
    return verdict, [results_path, report_path, requirements_path]


source_bytes = client.get_file(file_id=FILE_ID).revisions[-1].read_bytes()
verdict, output_paths = run_checks(source_bytes, work / "iter-1")

## 5. Upload the outputs and record the log entry

`create_workflow_output` registers each result file against the system; `create_workflow_log_entry` ties the title, `status`, `configuration_id`, and output IDs into one durable record. Both iterations use the same helper.

> **Pause here.** Open the system in the web app → **Workflow log** tab, open the failed entry, and preview the attached outputs — including the requirement set the run was judged against.

In [ ]:
def log_run(title, verdict, paths):
    """Upload each output file, then record one workflow log entry."""
    output_ids = [v3.create_workflow_output(system_id=SYSTEM_ID, path=p).id for p in paths]
    entry = v3.create_workflow_log_entry(
        system_id=SYSTEM_ID,
        workflow_log_entry_create_dto=WorkflowLogEntryCreateDto(
            title=title,
            status=verdict,  # SUCCESS | FAILED | UNSPECIFIED
            configuration_id=CONFIG_ID,
            workflow_output_ids=output_ids,
        ),
    )
    print(f"{entry.status}  {title}  ({len(output_ids)} outputs, entry {entry.id})")
    return entry


entry1 = log_run("Requirements verification - iter-1 (initial design)", verdict, output_paths)
print("Workflow log tab:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 6. Fix the violation, re-run, and log the pass

A thicker heat spreader brings the internal component temperature down to 104 °C, inside the 110 °C ceiling of requirement 3.6.1.1.2. Every other parameter is untouched. The verification code is unchanged — `get_file` returns the latest revision, so the second run picks up the new numbers.

In [ ]:
REVISED_DESIGN = {
    "COLDPLATE_1": {"length_in": 9.0, "width_in": 3.4, "height_in": 0.94},
    "THERMAL": {"coolant_temp_c": 65.0, "component_temp_c": 104.0, "power_module_temp_c": 132.0},
}
design.write_text(json.dumps(REVISED_DESIGN, indent=2))
client.update_model(
    model_id=MODEL_ID,
    path=design,
    description="HVMC coldplate + thermal budget - 3.6.1.1.2 fixed",
    version_name="v2-req-fixes",
)
commit_changes(client, SYSTEM_ID, CONFIG_ID)

source_bytes = client.get_file(file_id=FILE_ID).revisions[-1].read_bytes()
verdict, output_paths = run_checks(source_bytes, work / "iter-2")

entry2 = log_run("Requirements verification - iter-2 (3.6.1.1.2 fixed)", verdict, output_paths)
print("Review both entries:", f"{UI_URL}/systems/{SYSTEM_ID}")

## 7. Attach the verified results to the design model

`create_resource(resource_type=artifact)` uploads the passing report — the V3 Resources equivalent of `Client.add_artifact()` — and a `produces` revision relationship links the design model revision to it. Workflow outputs stay outside configuration snapshots; a model-linked artifact does not, so one more `commit_changes` puts it on the baseline snapshot.

In [ ]:
import shutil

from istari_digital_client.v3.models.new_revision_relationship_dto import NewRevisionRelationshipDto

# ResourceTypeDto was imported in section 2.
# The passing report from iter-2, renamed to say what it is.
verified = work / f"verified-results-for-{design.name}"
shutil.copyfile(work / "iter-2" / "results.json", verified)

artifact = v3.create_resource(
    path=verified,
    resource_type=ResourceTypeDto.ARTIFACT,
    display_name=verified.name,
    description=f"Verified against {REQ_SOURCE['cameo_model']} requirements (entry {entry2.id})",
)

model_resource = v3.get_resource(resource_id=MODEL_ID)
produces = next(t for t in v3.list_revision_relationship_types().items if t.name == "produces")
v3.create_revision_relationship(
    new_revision_relationship_dto=NewRevisionRelationshipDto(
        relationship_type_id=produces.id,
        left_revision_id=model_resource.file_revision_id,
        right_revision_id=artifact.file_revision_id,
    ),
)
commit_changes(client, SYSTEM_ID, CONFIG_ID)

print(f"Artifact {artifact.resource_id} linked to model {MODEL_ID} via {produces.name}")
print("Committed to the baseline snapshot")

## Recap

On one system you now have:

- Two workflow log entries — `FAILED`, then `SUCCESS` — each linked to the `requirements-check` configuration
- Every result file from both runs, including the exact `requirements.json` revision each verdict was judged against
- The verified report as an artifact resource on the design model via a `produces` relationship, committed to the baseline snapshot

The limits were never written down here. They live in the Cameo model, reach this notebook through `@istari:extract`, and each result carries the `req_id` it satisfies — so a requirement change in Cameo, re-extracted, flips the verdict on the next run with no code change.

### Learn more

- [External workflow logs](https://docs.istaridigital.com/developers/SDK/v3/03-workflow-logs) - `create_workflow_output`, `create_workflow_log_entry`, and listing entries
- [Cameo requirements extraction + tag update](../cameo_extract_and_update_notebook%20-%20demo%20(sdk%20only).ipynb) - producing `requirements.json` and writing values back with `@istari:update_tags`
- [Scenario A quickstart](workflow_log_scenario_a_simple.ipynb) - the same workflow-log flow with a synthetic check
- [Scenario B](workflow_log_scenario_b.ipynb) - logging a tradespace sweep

## Teardown

Archive the demo system so repeated runs do not clutter the instance. The Cameo model and its extraction are left untouched — this notebook only read them. Archiving is reversible.

In [ ]:
client.archive_system(system_id=SYSTEM_ID)
print("Archived system", SYSTEM_ID)